In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install supabase pyngrok -q
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
from supabase import create_client
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
import os
import numpy as np
import json
from pydantic import BaseModel
import nest_asyncio
import uvicorn
import asyncio
from fastapi import FastAPI
secrets = UserSecretsClient()
url = secrets.get_secret("SUPABASE_URL")      
key = secrets.get_secret("SUPABASE_KEY")      
supabase = create_client(url, key)
ngrok.set_auth_token(secrets.get_secret('NGROK_TOKEN'))
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

## Loading the LLM

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 4096, 
    dtype = None,          
    load_in_4bit = True,   
    device_map = {"": 1}
)

FastLanguageModel.for_inference(model)

print("14B Model Loaded Successfully on GPU 1!")

## Loading the embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(
    "/kaggle/input/datasets/lakshyaupadhyaya/lakshyaupadhyayanomic-embed-text-v1-5/nomic-embed",
    device="cuda:0",
    trust_remote_code=True
)

## Loading the sentiment model

In [ ]:
from transformers import pipeline
emotion_classifier = pipeline(
    "text-classification",
    model="samlowe/roberta-base-go_emotions",
    top_k=None,
    device=0
)

In [ ]:
from langchain_core.prompts import PromptTemplate
template = """<|im_start|>system
Transform the user's movie query into a structured format matching this template:
Title: <exact title or franchise name mentioned> | Description: <plot/theme/genre keywords> | Directed by: <director if explicitly named> | Starring: <actors if explicitly named> | Emotion: <mood/emotion keywords> | Language: <if explicitly mentioned>
Rules:
- ONLY include what the user explicitly mentioned, no inference
- If user says "marvel" put "Marvel" in Title, nothing else
- If user says "2026" skip it entirely, leave field out
- Do not expand, assume, or hallucinate any details
- Leave out any field you are not certain about
Examples:
"funny movies" -> "Description: comedy | Emotion: amusement joy"
"marvel movies 2026" -> "Title: Marvel"
"movies with the rock" -> "Starring: The Rock"
"sad romantic films" -> "Description: romance | Emotion: sadness"
"superhero action films" -> "Description: superhero action"
"hindi movies" -> "Language: Hindi"
<|im_end|>
<|im_start|>user
{query}
<|im_end|>
<|im_start|>assistant
"""

rewrite_prompt = PromptTemplate(input_variables=["query"], template=template)

async def rewrite_for_retrieval(query: str) -> str:
    FastLanguageModel.for_inference(model)
    prompt_string = rewrite_prompt.format(query=query)
    
    inputs = tokenizer([prompt_string], return_tensors="pt").to("cuda:1")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>")
    )
    
    rewritten = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return rewritten

In [ ]:
import re
async def extract_metadata(rewritten: str, original_query: str) -> dict:
    filters = {}
    
    field_map = {
        "Starring": "starring_filter",
        "Directed by": "director_filter",
        "Title": "title_filter",
        "Language": "language_filter"
    }
    
    for field, key in field_map.items():
        if f"{field}:" in rewritten:
            value = rewritten.split(f"{field}:")[1].split("|")[0].strip()
            if value:
                filters[key] = value
    
    years = re.findall(r'\b(?:19|20)\d{2}\b', original_query)
    if years:
        filters["year_filter"] = int(years[0])
    
    return filters

In [ ]:
import json
async def get_embedding(query, filters={}):
    def _sync():
        query_embedding = embedder.encode(["search_query: " + query])[0].tolist()
        
        params = {
            "query_embedding": query_embedding,
            "match_count": 10,
            **filters
        }
        
        return supabase.rpc("match_movies", params).execute()
    return await asyncio.to_thread(_sync)

In [ ]:
emotions_labels=["admiration","amusement","anger","annoyance","approval","caring",
"confusion","curiosity","desire","disappointment","disapproval",
"disgust","embarrassment","excitement","fear","gratitude","grief",
"joy","love","nervousness","optimism","pride","realization",
"relief","remorse","sadness","surprise","neutral"
]
def cosine_sim(q, m):
    return np.dot(q, m) / (np.linalg.norm(q) * np.linalg.norm(m))
def vectorizer(emotions):
    score={x["label"]:x["score"] for x in emotions}
    return [score.get(x,0.0) for x in emotions_labels]
def format_context(movies):
    result = ""
    for m in movies:
        result += f"Title: {m['title']}\n"
        result += f"Description: {m['description']}\n"
        result += f"Directed by: {m['directed_by']}\n"
        result += f"Starring: {m['starring']}\n"
    return result

async def get_ranked_context(results, query):
    def _sync():
        emotions = emotion_classifier(query)[0]
        query_vec = vectorizer(emotions)
        for movie in results.data:
            sentiments = movie['all_sentiments']
            # handle both string and already-parsed list
            if isinstance(sentiments, str):
                sentiments = json.loads(sentiments)
            movie['emotion_similarity'] = cosine_sim(query_vec, vectorizer(sentiments))
        reranked = sorted(results.data, key=lambda x: x["emotion_similarity"], reverse=True)
        return format_context(reranked[:3])
    return await asyncio.to_thread(_sync)
        

In [ ]:
from unsloth import FastLanguageModel


template = """<|im_start|>system
You are an enthusiastic, expert movie recommender. Your audience is a casual movie watcher looking for great suggestions. Your task is to recommend movies conversationally based STRICTLY on the retrieved context below.
Guidelines:
- Action: Write a flowing, engaging response that recommends the movies naturally. Do not just output a rigid list.
- Features: Highlight key selling points from the context, such as the director, the stars, the core plot, and the general vibe.
- Tone: Suggestive, helpful, and friendly.
- Constraints: Do not hallucinate or invent details. Use ONLY the information provided in the "Movies retrieved" section.
<|im_end|>
<|im_start|>user
Movies retrieved:
{context}

User query: {query}
<|im_end|>
<|im_start|>assistant
"""

generate_prompt = PromptTemplate(
    input_variables=["context", "query"],
    template=template
)

In [ ]:
from fastapi.middleware.cors import CORSMiddleware
from fastapi import Request
from fastapi.responses import StreamingResponse
from typing import Union
class ChatRequest(BaseModel):
    query: str
    history: Union[list,str]=""
    
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_methods=["*"], 
    allow_headers=["*"],  
)

@app.post("/chat")
async def chat(req: ChatRequest):
    print("Query:", req.query)
    
    rewritten = await rewrite_for_retrieval(req.query)
    print("Rewritten:", rewritten)
    
    # if rewrite model couldn't map to movie fields, it's off-topic
    values = re.findall(r"(?:Title|Description|Directed by|Starring|Emotion|Language):\s*([^|]+)", rewritten)
    if not any(v.strip() for v in values):
        return StreamingResponse(
            iter([f"data: {json.dumps({'token': 'I can only help with movie recommendations! Try asking me about a genre, actor, director, or mood.'})}\n\ndata: [DONE]\n\n"]),
            media_type="text/event-stream",
            headers={"X-Accel-Buffering": "no", "Cache-Control": "no-cache"}
        )
    
    filters = await extract_metadata(rewritten, req.query)
    
    filters = await extract_metadata(rewritten, req.query)
    print("Filters:", filters)
    
    embedding = await get_embedding(rewritten, filters)
    print("Embedding results count:", len(embedding.data))
    
    context = await get_ranked_context(embedding, req.query)
    print("Context being sent to LLM:\n", context)
    
    async def token_stream():
        FastLanguageModel.for_inference(model)
        prompt_string = generate_prompt.format(context=context, query=req.query)
        inputs = tokenizer([prompt_string], return_tensors="pt").to("cuda:1")
        
        from transformers import TextIteratorStreamer
        from threading import Thread
        
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        
        gen_kwargs = dict(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
            streamer=streamer,
        )
        
        thread = Thread(target=model.generate, kwargs=gen_kwargs)
        thread.start()
        
        for token in streamer:
            yield f"data: {json.dumps({'token': token})}\n\n"
        
        yield "data: [DONE]\n\n"
    
    return StreamingResponse(
        token_stream(),
        media_type="text/event-stream",
        headers={
            "X-Accel-Buffering": "no",
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
        }
    )

In [ ]:
import uvicorn
from pyngrok import ngrok

PORT = 8000

# 1. Setup the ngrok tunnel
public_url = ngrok.connect(PORT, bind_tls=True,domain="semiphosphorescent-unmortared-charlize.ngrok-free.dev")
print(f"\n{'='*55}")
print(f"  ngrok public URL : {public_url}")
print(f"  POST endpoint    : {public_url}/chat")
print(f"  Health check     : {public_url}/docs")
print(f"{'='*55}\n")

# 2. Configure the Uvicorn server manually
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)

# 3. Await it natively instead of calling .run()
await server.serve()